# DAZONE 2026 — TabNet Uplift Modeling Notebook

Bản này chuyển toàn bộ **base Machine Learning model** sang **TabNet** (`pytorch-tabnet`) và giữ nguyên hệ meta-learner/uplift learner đang có:

```python
RUN_LEARNERS = ["T", "S", "INT", "CT", "TO", "X", "R", "DR"]
RUN_MODEL_KINDS = ["tabnet"]
```

Mục tiêu vẫn là tối đa hóa:

```python
Qini_coefficient_total = Qini_coefficient_cash + Qini_coefficient_card
```

Với mỗi model, notebook vẫn in đủ 3 cột chính:

- `Qini_coefficient_cash`
- `Qini_coefficient_card`
- `Qini_coefficient_total`

Ghi chú logic quan trọng:

- T-Learner vẫn được chạy riêng cho `cash` và `card`.
- Khi chạy `cash`: TabNet fit hai response model `Cash-treatment + Cash-label` và `Card-treatment + Cash-label`.
- Khi chạy `card`: TabNet fit hai response model `Card-treatment + Card-label` và `Cash-treatment + Card-label`.
- S/X/R/DR/TO/CT/INT learner giữ nguyên logic cũ, chỉ thay lõi classifier/regressor thành TabNet.
- Direct uplift tree/RF và UTBoost được tắt mặc định vì chúng không phải TabNet, tránh sai yêu cầu “TabNet thay cho tất cả mô hình machine learning”.

---
### Cách chạy nhanh

Trên Kaggle, bật Internet nếu cần cài `pytorch-tabnet`, sau đó Run All.

Nếu runtime quá lâu, giảm:

```python
TABNET_MAX_EPOCHS = 40
TABNET_PATIENCE = 6
TABNET_MAX_TRAIN_ROWS = 50000
```

Nếu muốn tối đa accuracy và đủ thời gian/GPU, tăng:

```python
TABNET_MAX_EPOCHS = 120
TABNET_PATIENCE = 15
TABNET_MAX_TRAIN_ROWS = None
```


In [ ]:
# ============================================================
# KAGGLE ENVIRONMENT SETUP — TABNET VERSION
# ============================================================
# Trên Kaggle: bật Notebook Settings -> Internet -> On nếu muốn tự cài package còn thiếu.
# Nếu package đã có sẵn, cell này sẽ không cài lại.

RUN_INSTALL = True

import importlib.util
import subprocess
import sys
import warnings


def is_installed(import_name: str) -> bool:
    return importlib.util.find_spec(import_name) is not None


def pip_install(package: str):
    print(f"Installing: {package}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


if RUN_INSTALL:
    required = [
        ("numpy", "numpy"),
        ("pandas", "pandas"),
        ("scipy", "scipy"),
        ("sklearn", "scikit-learn"),
        ("matplotlib", "matplotlib"),
        ("torch", "torch"),
        ("pytorch_tabnet", "pytorch-tabnet"),
    ]

    for import_name, package_name in required:
        if not is_installed(import_name):
            try:
                pip_install(package_name)
            except Exception as e:
                print(f"WARNING: Không cài được {package_name}: {repr(e)}")
                print("Notebook sẽ dừng ở cell import nếu package bắt buộc chưa có.")
        else:
            print(f"OK: {import_name} already installed")
else:
    print("RUN_INSTALL=False -> bỏ qua cài package.")

warnings.filterwarnings("ignore")


In [ ]:
import gc
import json
import os
import sys
import time
import warnings
from pathlib import Path

# Tránh OpenMP/BLAS dùng quá nhiều thread khiến sklearn/torch chạy chậm trên Kaggle.
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "4")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore")

SEED = 42
N_SPLITS = 5
N_JOBS = 4

# ============================================================
# PRESET TABNET
# ============================================================
# Giữ nguyên learner, chỉ thay toàn bộ base ML model bằng TabNet.
RUN_PRESET = "TABNET"
RUN_MODEL_KINDS = ["tabnet"]
ALL_UPLIFT_LEARNERS = ["T", "S", "INT", "CT", "TO", "X", "R", "DR"]
RUN_LEARNERS = ALL_UPLIFT_LEARNERS

# ============================================================
# TABNET HYPERPARAMETERS
# ============================================================
# Cấu hình này cân bằng accuracy/runtime cho Kaggle GPU T4/P100.
# Muốn nhanh hơn: giảm TABNET_MAX_EPOCHS, TABNET_PATIENCE hoặc đặt TABNET_MAX_TRAIN_ROWS=50000.
# Muốn tận dụng full data: giữ TABNET_MAX_TRAIN_ROWS=None.
TABNET_N_D = 32
TABNET_N_A = 32
TABNET_N_STEPS = 5
TABNET_GAMMA = 1.5
TABNET_N_INDEPENDENT = 2
TABNET_N_SHARED = 2
TABNET_LAMBDA_SPARSE = 1e-4
TABNET_MASK_TYPE = "entmax"       # "sparsemax" cũng có thể thử, nhưng entmax thường ổn định hơn.
TABNET_LR = 2e-2
TABNET_WEIGHT_DECAY = 1e-5
TABNET_MAX_EPOCHS = 80
TABNET_PATIENCE = 10
TABNET_BATCH_SIZE = 2048
TABNET_VIRTUAL_BATCH_SIZE = 256
TABNET_INTERNAL_VAL_SIZE = 0.15   # split từ train fold, không dùng validation fold ngoài -> không leakage.
TABNET_MAX_TRAIN_ROWS = None      # None = dùng toàn bộ train fold. Có thể đặt 50000 nếu runtime quá lâu.
TABNET_DEVICE_NAME = "auto"       # "auto", "cuda", hoặc "cpu".
TABNET_VERBOSE = 0

# Direct uplift tree/RF/UTBoost không còn chạy trong bản TabNet-only.
RUN_OPTIONAL_CAUSALML_TREE = False
RUN_OPTIONAL_UTBOOST = False
CAUSALML_TREE_CRITERIA = []
CAUSALML_FOREST_CRITERIA = []
UTB_ITERATIONS = 0

TOPK_ENSEMBLE = 8

print("Python:", sys.version.split()[0])
print("Preset:", RUN_PRESET)
print("N_SPLITS:", N_SPLITS, "| N_JOBS:", N_JOBS)
print("Run base models:", RUN_MODEL_KINDS)
print("Learners:", RUN_LEARNERS)
print("Optional CausalML tree/RF:", RUN_OPTIONAL_CAUSALML_TREE)
print("Optional UTBoost:", RUN_OPTIONAL_UTBOOST)
print("TabNet config:", {
    "n_d": TABNET_N_D,
    "n_a": TABNET_N_A,
    "n_steps": TABNET_N_STEPS,
    "gamma": TABNET_GAMMA,
    "max_epochs": TABNET_MAX_EPOCHS,
    "patience": TABNET_PATIENCE,
    "batch_size": TABNET_BATCH_SIZE,
    "virtual_batch_size": TABNET_VIRTUAL_BATCH_SIZE,
    "internal_val_size": TABNET_INTERNAL_VAL_SIZE,
    "max_train_rows": TABNET_MAX_TRAIN_ROWS,
    "device_name": TABNET_DEVICE_NAME,
})


In [ ]:
AVAILABLE = {}
warnings.filterwarnings("ignore")

try:
    import sklearn
    from sklearn.base import BaseEstimator, ClassifierMixin, RegressorMixin
    from sklearn.model_selection import train_test_split
    print("scikit-learn:", sklearn.__version__)
except Exception as e:
    raise ImportError(f"scikit-learn is required for KFold/splitting/wrappers: {repr(e)}")

try:
    import torch
    from pytorch_tabnet.tab_model import TabNetClassifier, TabNetRegressor
    AVAILABLE["tabnet"] = True
    print("torch:", torch.__version__)
    print("pytorch-tabnet: available")
except Exception as e:
    AVAILABLE["tabnet"] = False
    raise ImportError(
        "Không import được pytorch-tabnet. Trên Kaggle hãy bật Internet và chạy cell install: "
        "pip install pytorch-tabnet. Lỗi gốc: " + repr(e)
    )

# Các package direct uplift model bị tắt trong bản TabNet-only.
AVAILABLE["causalml"] = False
AVAILABLE["utboost"] = False

RUN_MODEL_KINDS = [k for k in RUN_MODEL_KINDS if AVAILABLE.get(k, False)]
if not RUN_MODEL_KINDS:
    raise ImportError("Không có TabNet base learner khả dụng. Cần cài pytorch-tabnet.")

print("Run base models after availability filter:", RUN_MODEL_KINDS)
print("Total base models:", len(RUN_MODEL_KINDS))


## 1. Load dữ liệu

In [ ]:

def find_file_pair(feature_name="df_feature.csv", target_name="df_target.csv"):
    roots = []
    for p in [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd(), Path.cwd() / "data", Path("/mnt/data")]:
        if p.exists() and str(p.resolve()) not in ["/", "/mnt"]:
            roots.append(p)

    candidates = []
    for root in roots:
        direct_feat = root / feature_name
        direct_tgt = root / target_name
        if direct_feat.exists() and direct_tgt.exists():
            return direct_feat, direct_tgt
        try:
            for feat_path in root.rglob(feature_name):
                tgt_path = feat_path.parent / target_name
                if tgt_path.exists():
                    candidates.append((feat_path, tgt_path))
        except Exception:
            pass

    if candidates:
        candidates = sorted(candidates, key=lambda x: len(str(x[0])))
        return candidates[0]
    raise FileNotFoundError("Không tìm thấy df_feature.csv và df_target.csv")

feat_path, tgt_path = find_file_pair()
print("Feature path:", feat_path)
print("Target path :", tgt_path)

feat = pd.read_csv(feat_path)
tgt = pd.read_csv(tgt_path)
assert len(feat) == len(tgt), "df_feature và df_target phải cùng số dòng"

TARGET_CASH = "cash_sign_30d"
TARGET_CARD = "card_sign_30d"
TREAT_COL_BINARY = "treatment_cash"
TREAT_COL_RAW = "treatment"

assert TARGET_CASH in tgt.columns and TARGET_CARD in tgt.columns, "Thiếu target cash/card"

if TREAT_COL_BINARY in feat.columns:
    W = feat[TREAT_COL_BINARY].astype(int).to_numpy()  # 1 = Cash, 0 = Card
    TREAT_COL = TREAT_COL_BINARY
elif TREAT_COL_RAW in feat.columns:
    W = feat[TREAT_COL_RAW].astype(str).str.lower().map({"cash": 1, "card": 0}).astype(int).to_numpy()
    TREAT_COL = TREAT_COL_RAW
else:
    raise ValueError("Thiếu cột treatment_cash hoặc treatment")

Y_cash = tgt[TARGET_CASH].astype(int).to_numpy()
Y_card = tgt[TARGET_CARD].astype(int).to_numpy()

id_cols = [c for c in ["customer_id", "id", "Customer_ID"] if c in feat.columns]
ID_COL = id_cols[0] if id_cols else None

DROP_COLS = [TREAT_COL]
if ID_COL:
    DROP_COLS.append(ID_COL)
FEAT_COLS = [c for c in feat.columns if c not in DROP_COLS]

X_df = feat[FEAT_COLS].copy()
X = X_df.to_numpy(dtype=np.float32)

w_cash = (W == 1).astype(int)
w_card = (W == 0).astype(int)

print(f"N={len(X):,} | features={X.shape[1]}")
print(f"Treatment Cash: {W.sum():,} ({W.mean():.2%}) | Card: {(1-W).sum():,} ({(1-W).mean():.2%})")
print(f"Cash target rate: {Y_cash.mean():.4%} | Card target rate: {Y_card.mean():.4%}")
print("Feature NaN total:", int(pd.isna(X_df).sum().sum()))


In [ ]:

def find_optional_test_feature():
    names = ["df_feature_test.csv", "test_feature.csv", "X_test.csv", "test.csv"]
    roots = [p for p in [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd(), Path.cwd() / "data", Path("/mnt/data")]
             if p.exists() and str(p.resolve()) not in ["/", "/mnt"]]
    for root in roots:
        for name in names:
            direct = root / name
            if direct.exists():
                return direct
            try:
                hits = list(root.rglob(name))
                if hits:
                    return sorted(hits, key=lambda p: len(str(p)))[0]
            except Exception:
                pass
    return None

test_path = find_optional_test_feature()
if test_path:
    test_feat = pd.read_csv(test_path)
    test_id = test_feat[ID_COL].copy() if ID_COL and ID_COL in test_feat.columns else pd.Series(np.arange(len(test_feat)), name="row_id")
    missing = [c for c in FEAT_COLS if c not in test_feat.columns]
    extra = [c for c in test_feat.columns if c not in FEAT_COLS + ([ID_COL] if ID_COL else [])]
    if missing:
        raise ValueError(f"Test thiếu feature: {missing[:10]}")
    X_test = test_feat[FEAT_COLS].to_numpy(dtype=np.float32)
    print("Test feature path:", test_path)
    print("Test N:", len(X_test), "| extra columns ignored:", extra[:10])
else:
    X_test = None
    test_id = None
    print("Không thấy test feature riêng. Notebook sẽ xuất OOF score trên train.")


## 2. Metric Qini + fold cố định

In [ ]:

_trapz = np.trapezoid if hasattr(np, "trapezoid") else np.trapz


def qini_curve(y, w_event, score):
    y = np.asarray(y, dtype=float)
    w_event = np.asarray(w_event, dtype=int)
    score = np.asarray(score, dtype=float)
    order = np.argsort(-score, kind="mergesort")
    y_s = y[order]
    w_s = w_event[order]
    n_t = max(int(w_event.sum()), 1)
    n_c = max(len(w_event) - n_t, 1)
    q = np.cumsum(y_s * w_s) / n_t - np.cumsum(y_s * (1 - w_s)) / n_c
    x = np.arange(1, len(y) + 1) / len(y)
    return np.r_[0.0, x], np.r_[0.0, q]


def oracle_score(y, w_event):
    y = np.asarray(y, dtype=int)
    w_event = np.asarray(w_event, dtype=int)
    return w_event * y - (1 - w_event) * y


def qini_metrics(y, w_event, score):
    x, q = qini_curve(y, w_event, score)
    auc_model = float(_trapz(q, x))
    ate = float(q[-1])
    auc_random = ate * 0.5
    _, q_perfect = qini_curve(y, w_event, oracle_score(y, w_event))
    auc_perfect = float(_trapz(q_perfect, x))
    qini_diff = auc_model - auc_random
    denom = auc_perfect - auc_random
    qini_norm = qini_diff / denom if abs(denom) > 1e-12 else np.nan
    qini_ratio = auc_model / auc_random if abs(auc_random) > 1e-12 else np.nan
    return {
        "ate": ate,
        "auc_model": auc_model,
        "auc_random": auc_random,
        "auc_perfect": auc_perfect,
        "qini_diff": qini_diff,
        "qini_norm": qini_norm,
        "qini_ratio": qini_ratio,
    }


def stratify_key(w_event, y):
    return (np.asarray(w_event, dtype=int) * 2 + np.asarray(y, dtype=int)).astype(int)


def make_fold_id(w_event, y):
    key = stratify_key(w_event, y)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    fold_id = np.zeros(len(y), dtype=int)
    dummy_x = np.zeros(len(y))
    for fold, (_, va) in enumerate(skf.split(dummy_x, key), 1):
        fold_id[va] = fold
    return fold_id


def rank01(a):
    a = np.asarray(a, dtype=float)
    return rankdata(a, method="average") / len(a)


def rank_average(score_list, weights=None):
    score_list = [np.asarray(s, dtype=float) for s in score_list if s is not None]
    if not score_list:
        return None
    ranks = np.vstack([rank01(s) for s in score_list])
    if weights is None:
        return ranks.mean(axis=0)
    weights = np.asarray(weights, dtype=float)
    if len(weights) != len(score_list) or weights.sum() <= 0:
        return ranks.mean(axis=0)
    weights = weights / weights.sum()
    return np.average(ranks, axis=0, weights=weights)


fold_ids = {
    "cash": make_fold_id(w_cash, Y_cash),
    "card": make_fold_id(w_card, Y_card),
}

print("Fold distribution cash target:", pd.Series(fold_ids["cash"]).value_counts().sort_index().to_dict())
print("Fold distribution card target:", pd.Series(fold_ids["card"]).value_counts().sort_index().to_dict())


## 3. Model factory — TabNet-only

Toàn bộ base classifier/regressor được chuyển sang TabNet:

- `TabNetSklearnClassifier` dùng cho outcome nhị phân trong T/S/INT/CT/X/R/DR.
- `TabNetSklearnRegressor` dùng cho pseudo-outcome liên tục trong TO/X/R/DR.

Wrapper có internal validation split **chỉ từ train fold**, nên không dùng validation fold ngoài và không tạo CV leakage.


In [ ]:
class ConstantClassifier:
    def __init__(self, value):
        self.value = float(value)
        self.classes_ = np.array([0, 1])
    def fit(self, X, y, sample_weight=None):
        return self
    def predict_proba(self, X):
        p = np.full(len(X), self.value, dtype=float)
        p = np.clip(p, 0.0, 1.0)
        return np.c_[1 - p, p]


class ConstantRegressor:
    def __init__(self, value):
        self.value = float(value)
    def fit(self, X, y, sample_weight=None):
        return self
    def predict(self, X):
        return np.full(len(X), self.value, dtype=float)


def sigmoid_safe(z):
    z = np.clip(np.asarray(z, dtype=float), -35, 35)
    return 1.0 / (1.0 + np.exp(-z))


def to_float32_matrix(X):
    """TabNet cần dense float32 và không chịu NaN/Inf."""
    X = np.asarray(X, dtype=np.float32)
    if not np.isfinite(X).all():
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    return X


def stratified_subsample_indices(y, max_rows, seed=SEED):
    y = np.asarray(y)
    n = len(y)
    if max_rows is None or n <= int(max_rows):
        return np.arange(n)
    max_rows = int(max_rows)
    rng = np.random.default_rng(seed)
    # Nếu binary thì giữ tỷ lệ class tốt hơn để TabNet không mất positive class.
    uniq = np.unique(y.astype(int)) if np.all(np.isfinite(y)) else np.unique(y)
    if len(uniq) <= 2 and set(np.unique(y.astype(int))).issubset({0, 1}):
        pos_idx = np.where(y.astype(int) == 1)[0]
        neg_idx = np.where(y.astype(int) == 0)[0]
        pos_ratio = len(pos_idx) / max(n, 1)
        keep_pos = min(len(pos_idx), max(1, int(max_rows * pos_ratio))) if len(pos_idx) else 0
        keep_neg = min(len(neg_idx), max_rows - keep_pos)
        # Đảm bảo giữ positive nhiều nhất có thể nếu positive hiếm.
        if len(pos_idx) > 0 and keep_pos < min(len(pos_idx), max_rows // 2):
            keep_pos = min(len(pos_idx), max_rows // 2)
            keep_neg = min(len(neg_idx), max_rows - keep_pos)
        pos_sel = rng.choice(pos_idx, size=keep_pos, replace=False) if len(pos_idx) > keep_pos else pos_idx
        neg_sel = rng.choice(neg_idx, size=keep_neg, replace=False) if len(neg_idx) > keep_neg else neg_idx
        sel = np.r_[pos_sel, neg_sel]
        rng.shuffle(sel)
        return sel
    sel = rng.choice(np.arange(n), size=max_rows, replace=False)
    return np.sort(sel)


def maybe_clear_torch_cache():
    try:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


class TabNetSklearnClassifier(BaseEstimator, ClassifierMixin):
    """Sklearn-like binary classifier wrapper cho pytorch-tabnet."""
    def __init__(self, seed=SEED):
        self.seed = seed
        self.classes_ = np.array([0, 1])

    def _make_model(self):
        return TabNetClassifier(
            n_d=TABNET_N_D,
            n_a=TABNET_N_A,
            n_steps=TABNET_N_STEPS,
            gamma=TABNET_GAMMA,
            n_independent=TABNET_N_INDEPENDENT,
            n_shared=TABNET_N_SHARED,
            lambda_sparse=TABNET_LAMBDA_SPARSE,
            optimizer_fn=torch.optim.AdamW,
            optimizer_params=dict(lr=TABNET_LR, weight_decay=TABNET_WEIGHT_DECAY),
            scheduler_fn=torch.optim.lr_scheduler.ReduceLROnPlateau,
            scheduler_params=dict(mode="max", patience=3, factor=0.5, min_lr=1e-5),
            mask_type=TABNET_MASK_TYPE,
            seed=int(self.seed),
            device_name=TABNET_DEVICE_NAME,
            verbose=TABNET_VERBOSE,
        )

    def _train_valid_split(self, X, y, sample_weight):
        # Validation nội bộ chỉ lấy từ train fold hiện tại -> fold-safe.
        use_val = (
            TABNET_INTERNAL_VAL_SIZE is not None
            and float(TABNET_INTERNAL_VAL_SIZE) > 0
            and len(y) >= 100
            and len(np.unique(y)) == 2
            and np.min(np.bincount(y.astype(int), minlength=2)) >= 3
        )
        if not use_val:
            return X, None, y, None, sample_weight, None
        idx = np.arange(len(y))
        tr_idx, va_idx = train_test_split(
            idx,
            test_size=float(TABNET_INTERNAL_VAL_SIZE),
            random_state=int(self.seed),
            stratify=y.astype(int),
        )
        sw_tr = sample_weight[tr_idx] if sample_weight is not None else None
        sw_va = sample_weight[va_idx] if sample_weight is not None else None
        return X[tr_idx], X[va_idx], y[tr_idx], y[va_idx], sw_tr, sw_va

    def fit(self, X, y, sample_weight=None):
        X = to_float32_matrix(X)
        y = np.asarray(y, dtype=np.int64).reshape(-1)
        if sample_weight is not None:
            sample_weight = np.asarray(sample_weight, dtype=np.float32).reshape(-1)

        idx = stratified_subsample_indices(y, TABNET_MAX_TRAIN_ROWS, self.seed)
        X = X[idx]
        y = y[idx]
        if sample_weight is not None:
            sample_weight = sample_weight[idx]

        X_tr, X_va, y_tr, y_va, sw_tr, _ = self._train_valid_split(X, y, sample_weight)
        self.model_ = self._make_model()

        fit_kwargs = dict(
            X_train=X_tr,
            y_train=y_tr,
            max_epochs=int(TABNET_MAX_EPOCHS),
            patience=int(TABNET_PATIENCE),
            batch_size=int(TABNET_BATCH_SIZE),
            virtual_batch_size=int(TABNET_VIRTUAL_BATCH_SIZE),
            num_workers=0,
            drop_last=False,
        )
        if X_va is not None:
            fit_kwargs.update(
                eval_set=[(X_va, y_va)],
                eval_name=["valid"],
                eval_metric=["auc"],
            )
        if sw_tr is not None:
            fit_kwargs["weights"] = sw_tr

        try:
            self.model_.fit(**fit_kwargs)
        except TypeError:
            # Một số version pytorch-tabnet không nhận ndarray weights trong mọi task.
            fit_kwargs.pop("weights", None)
            self.model_.fit(**fit_kwargs)

        self.classes_ = np.array([0, 1])
        maybe_clear_torch_cache()
        return self

    def predict_proba(self, X):
        X = to_float32_matrix(X)
        proba = self.model_.predict_proba(X)
        proba = np.asarray(proba, dtype=float)
        if proba.ndim == 1:
            p = np.clip(proba, 0.0, 1.0)
            return np.c_[1.0 - p, p]
        if proba.shape[1] == 1:
            p = np.clip(proba[:, 0], 0.0, 1.0)
            return np.c_[1.0 - p, p]
        return proba[:, :2]

    def __del__(self):
        maybe_clear_torch_cache()


class TabNetSklearnRegressor(BaseEstimator, RegressorMixin):
    """Sklearn-like regressor wrapper cho pseudo-outcome liên tục."""
    def __init__(self, seed=SEED):
        self.seed = seed

    def _make_model(self):
        return TabNetRegressor(
            n_d=TABNET_N_D,
            n_a=TABNET_N_A,
            n_steps=TABNET_N_STEPS,
            gamma=TABNET_GAMMA,
            n_independent=TABNET_N_INDEPENDENT,
            n_shared=TABNET_N_SHARED,
            lambda_sparse=TABNET_LAMBDA_SPARSE,
            optimizer_fn=torch.optim.AdamW,
            optimizer_params=dict(lr=TABNET_LR, weight_decay=TABNET_WEIGHT_DECAY),
            scheduler_fn=torch.optim.lr_scheduler.ReduceLROnPlateau,
            scheduler_params=dict(mode="min", patience=3, factor=0.5, min_lr=1e-5),
            mask_type=TABNET_MASK_TYPE,
            seed=int(self.seed),
            device_name=TABNET_DEVICE_NAME,
            verbose=TABNET_VERBOSE,
        )

    def _train_valid_split(self, X, y, sample_weight):
        use_val = TABNET_INTERNAL_VAL_SIZE is not None and float(TABNET_INTERNAL_VAL_SIZE) > 0 and len(y) >= 100
        if not use_val:
            return X, None, y, None, sample_weight, None
        idx = np.arange(len(y))
        tr_idx, va_idx = train_test_split(
            idx,
            test_size=float(TABNET_INTERNAL_VAL_SIZE),
            random_state=int(self.seed),
            shuffle=True,
        )
        sw_tr = sample_weight[tr_idx] if sample_weight is not None else None
        sw_va = sample_weight[va_idx] if sample_weight is not None else None
        return X[tr_idx], X[va_idx], y[tr_idx], y[va_idx], sw_tr, sw_va

    def fit(self, X, y, sample_weight=None):
        X = to_float32_matrix(X)
        y = np.asarray(y, dtype=np.float32).reshape(-1, 1)
        y_flat = y.reshape(-1)
        if sample_weight is not None:
            sample_weight = np.asarray(sample_weight, dtype=np.float32).reshape(-1)

        idx = stratified_subsample_indices(y_flat, TABNET_MAX_TRAIN_ROWS, self.seed)
        X = X[idx]
        y = y[idx]
        if sample_weight is not None:
            sample_weight = sample_weight[idx]

        X_tr, X_va, y_tr, y_va, sw_tr, _ = self._train_valid_split(X, y, sample_weight)
        self.model_ = self._make_model()

        fit_kwargs = dict(
            X_train=X_tr,
            y_train=y_tr,
            max_epochs=int(TABNET_MAX_EPOCHS),
            patience=int(TABNET_PATIENCE),
            batch_size=int(TABNET_BATCH_SIZE),
            virtual_batch_size=int(TABNET_VIRTUAL_BATCH_SIZE),
            num_workers=0,
            drop_last=False,
        )
        if X_va is not None:
            fit_kwargs.update(
                eval_set=[(X_va, y_va)],
                eval_name=["valid"],
                eval_metric=["rmse"],
            )
        if sw_tr is not None:
            fit_kwargs["weights"] = sw_tr

        try:
            self.model_.fit(**fit_kwargs)
        except TypeError:
            fit_kwargs.pop("weights", None)
            self.model_.fit(**fit_kwargs)

        maybe_clear_torch_cache()
        return self

    def predict(self, X):
        X = to_float32_matrix(X)
        pred = np.asarray(self.model_.predict(X), dtype=float)
        return pred.reshape(-1)

    def __del__(self):
        maybe_clear_torch_cache()


def make_classifier(kind, seed=SEED):
    if kind == "tabnet":
        return TabNetSklearnClassifier(seed=seed)
    raise ValueError(f"Unknown classifier kind: {kind}")


def make_regressor(kind, seed=SEED):
    if kind == "tabnet":
        return TabNetSklearnRegressor(seed=seed)
    raise ValueError(f"Unknown regressor kind: {kind}")


def class_weights(y, pos_weight=None):
    y = np.asarray(y, dtype=int)
    if pos_weight is None:
        pos = max(int(y.sum()), 1)
        neg = max(len(y) - pos, 1)
        pos_weight = neg / pos
    return np.where(y == 1, pos_weight, 1.0).astype(np.float32)


def common_pos_weight(y_treat, y_ctrl):
    def spw(y):
        y = np.asarray(y, dtype=int)
        pos = max(int(y.sum()), 1)
        neg = max(len(y) - pos, 1)
        return neg / pos
    return 0.5 * (spw(y_treat) + spw(y_ctrl))


def fit_with_optional_sample_weight(model, X_train, y_train, sample_weight=None):
    X_train = to_float32_matrix(X_train)
    if sample_weight is None:
        model.fit(X_train, y_train)
        return model
    sample_weight = np.asarray(sample_weight, dtype=np.float32)
    try:
        model.fit(X_train, y_train, sample_weight=sample_weight)
        return model
    except TypeError:
        model.fit(X_train, y_train)
        return model
    except ValueError as e:
        if "sample_weight" in str(e) or "weights" in str(e):
            model.fit(X_train, y_train)
            return model
        raise


def safe_fit_classifier(kind, X_train, y_train, sample_weight=None, seed=SEED):
    y_train = np.asarray(y_train, dtype=int)
    if len(y_train) == 0 or len(np.unique(y_train)) < 2:
        return ConstantClassifier(float(y_train.mean()) if len(y_train) else 0.0)
    model = make_classifier(kind, seed=seed)
    return fit_with_optional_sample_weight(model, X_train, y_train, sample_weight=sample_weight)


def safe_fit_regressor(kind, X_train, y_train, sample_weight=None, seed=SEED):
    y_train = np.asarray(y_train, dtype=float)
    if len(y_train) == 0 or np.nanstd(y_train) < 1e-12:
        return ConstantRegressor(float(np.nanmean(y_train)) if len(y_train) else 0.0)
    model = make_regressor(kind, seed=seed)
    return fit_with_optional_sample_weight(model, X_train, y_train, sample_weight=sample_weight)


def predict_classifier(model, X_pred):
    X_pred = to_float32_matrix(X_pred)
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_pred)
        proba = np.asarray(proba)
        if proba.ndim == 1:
            return np.clip(proba.astype(float), 0.0, 1.0)
        if proba.shape[1] == 1:
            return np.clip(proba[:, 0].astype(float), 0.0, 1.0)
        return proba[:, 1].astype(float)
    if hasattr(model, "decision_function"):
        return sigmoid_safe(model.decision_function(X_pred))
    pred = model.predict(X_pred)
    return np.asarray(pred, dtype=float)


def predict_regressor(model, X_pred):
    X_pred = to_float32_matrix(X_pred)
    pred = np.asarray(model.predict(X_pred), dtype=float)
    if pred.ndim > 1:
        pred = pred[:, 0]
    return pred


## 4. Uplift learner functions

Giữ nguyên 8 learner: `T`, `S`, `INT`, `CT`, `TO`, `X`, `R`, `DR`.

Trong bản này, mọi lời gọi `safe_fit_classifier()` và `safe_fit_regressor()` đều tạo TabNet model. Direct uplift tree/RF và UTBoost vẫn còn function để không phá cấu trúc notebook cũ, nhưng đã tắt mặc định vì không phải TabNet.


In [ ]:

def fold_iter(target_name):
    fid = fold_ids[target_name]
    for fold in range(1, N_SPLITS + 1):
        va = np.where(fid == fold)[0]
        tr = np.where(fid != fold)[0]
        yield fold, tr, va


def get_target_arrays(target_name):
    if target_name == "cash":
        return Y_cash, w_cash, 1, "CASH"
    if target_name == "card":
        return Y_card, w_card, 0, "CARD"
    raise ValueError(target_name)


def print_fold_metric(prefix, target_name, y, w_event, va, pred):
    met = qini_metrics(y[va], w_event[va], pred)
    print(f"  fold {prefix}: Qini_coefficient_{target_name}={met['qini_norm']:+.5f} | Q_ratio={met['qini_ratio']:+.4f}")
    return met


def run_t_learner(kind, target_name, X_test=None):
    y, w_event, treat_value, label = get_target_arrays(target_name)
    oof = np.zeros(len(X), dtype=float)
    test_score = np.zeros(len(X_test), dtype=float) if X_test is not None else None
    print(f"\n[{label}] T-Learner {kind}")
    if target_name == "cash":
        print("  internal response models: Cash-treatment + Cash-label, Card-treatment + Cash-label")
    else:
        print("  internal response models: Card-treatment + Card-label, Cash-treatment + Card-label")

    for fold, tr, va in fold_iter(target_name):
        mask_t = w_event[tr] == 1
        mask_c = ~mask_t
        y_t = y[tr][mask_t]
        y_c = y[tr][mask_c]
        spw_common = common_pos_weight(y_t, y_c)
        m_t = safe_fit_classifier(kind, X[tr][mask_t], y_t, class_weights(y_t, spw_common), seed=SEED + fold)
        m_c = safe_fit_classifier(kind, X[tr][mask_c], y_c, class_weights(y_c, spw_common), seed=SEED + 100 + fold)
        pred = predict_classifier(m_t, X[va]) - predict_classifier(m_c, X[va])
        oof[va] = pred
        if X_test is not None:
            test_score += (predict_classifier(m_t, X_test) - predict_classifier(m_c, X_test)) / N_SPLITS
        print_fold_metric(fold, target_name, y, w_event, va, pred)
        del m_t, m_c
        gc.collect()
    full_met = qini_metrics(y, w_event, oof)
    print(f"  OOF: Qini_coefficient_{target_name}={full_met['qini_norm']:+.5f} | Q_ratio={full_met['qini_ratio']:+.4f}")
    return oof, test_score


def run_s_learner(kind, target_name, X_test=None):
    y, w_event, treat_value, label = get_target_arrays(target_name)
    oof = np.zeros(len(X), dtype=float)
    test_score = np.zeros(len(X_test), dtype=float) if X_test is not None else None
    print(f"\n[{label}] S-Learner {kind}")
    for fold, tr, va in fold_iter(target_name):
        Xtr_aug = np.c_[X[tr], w_event[tr].astype(np.float32)]
        Xva_event = np.c_[X[va], np.ones(len(va), dtype=np.float32)]
        Xva_control = np.c_[X[va], np.zeros(len(va), dtype=np.float32)]
        sw = class_weights(y[tr])
        m = safe_fit_classifier(kind, Xtr_aug, y[tr], sw, seed=SEED + fold)
        pred = predict_classifier(m, Xva_event) - predict_classifier(m, Xva_control)
        oof[va] = pred
        if X_test is not None:
            Xte_event = np.c_[X_test, np.ones(len(X_test), dtype=np.float32)]
            Xte_control = np.c_[X_test, np.zeros(len(X_test), dtype=np.float32)]
            test_score += (predict_classifier(m, Xte_event) - predict_classifier(m, Xte_control)) / N_SPLITS
        print_fold_metric(fold, target_name, y, w_event, va, pred)
        del m
        gc.collect()
    full_met = qini_metrics(y, w_event, oof)
    print(f"  OOF: Qini_coefficient_{target_name}={full_met['qini_norm']:+.5f} | Q_ratio={full_met['qini_ratio']:+.4f}")
    return oof, test_score


def run_interaction_learner(kind, target_name, X_test=None):
    # Strong S-learner variant: one model with treatment flag and X*treatment interactions.
    y, w_event, treat_value, label = get_target_arrays(target_name)
    oof = np.zeros(len(X), dtype=float)
    test_score = np.zeros(len(X_test), dtype=float) if X_test is not None else None
    print(f"\n[{label}] Interaction S-Learner {kind}")
    for fold, tr, va in fold_iter(target_name):
        wt = w_event[tr].astype(np.float32)[:, None]
        Xtr_aug = np.c_[X[tr], wt, X[tr] * wt]
        Xva_event = np.c_[X[va], np.ones((len(va), 1), dtype=np.float32), X[va]]
        Xva_control = np.c_[X[va], np.zeros((len(va), 1), dtype=np.float32), np.zeros_like(X[va])]
        sw = class_weights(y[tr])
        m = safe_fit_classifier(kind, Xtr_aug, y[tr], sw, seed=SEED + fold)
        pred = predict_classifier(m, Xva_event) - predict_classifier(m, Xva_control)
        oof[va] = pred
        if X_test is not None:
            Xte_event = np.c_[X_test, np.ones((len(X_test), 1), dtype=np.float32), X_test]
            Xte_control = np.c_[X_test, np.zeros((len(X_test), 1), dtype=np.float32), np.zeros_like(X_test)]
            test_score += (predict_classifier(m, Xte_event) - predict_classifier(m, Xte_control)) / N_SPLITS
        print_fold_metric(fold, target_name, y, w_event, va, pred)
        del m
        gc.collect()
    full_met = qini_metrics(y, w_event, oof)
    print(f"  OOF: Qini_coefficient_{target_name}={full_met['qini_norm']:+.5f} | Q_ratio={full_met['qini_ratio']:+.4f}")
    return oof, test_score


def run_class_transformation(kind, target_name, X_test=None):
    # Revert-label / Class Transformation: z=1 nếu treated&converted hoặc control&not converted.
    y, w_event, treat_value, label = get_target_arrays(target_name)
    z = np.where(w_event == 1, y, 1 - y).astype(int)
    oof = np.zeros(len(X), dtype=float)
    test_score = np.zeros(len(X_test), dtype=float) if X_test is not None else None
    print(f"\n[{label}] Class Transformation {kind}")
    for fold, tr, va in fold_iter(target_name):
        e = np.clip(w_event[tr].mean(), 1e-6, 1 - 1e-6)
        balance_w = np.where(w_event[tr] == 1, 1.0 / e, 1.0 / (1.0 - e)).astype(np.float32)
        sw = balance_w * class_weights(z[tr])
        m = safe_fit_classifier(kind, X[tr], z[tr], sw, seed=SEED + fold)
        pred = 2.0 * predict_classifier(m, X[va]) - 1.0
        oof[va] = pred
        if X_test is not None:
            test_score += (2.0 * predict_classifier(m, X_test) - 1.0) / N_SPLITS
        print_fold_metric(fold, target_name, y, w_event, va, pred)
        del m
        gc.collect()
    full_met = qini_metrics(y, w_event, oof)
    print(f"  OOF: Qini_coefficient_{target_name}={full_met['qini_norm']:+.5f} | Q_ratio={full_met['qini_ratio']:+.4f}")
    return oof, test_score


def run_transformed_outcome(kind, target_name, X_test=None):
    # y* = W*Y/e - (1-W)*Y/(1-e), E[y*|X] = treatment effect.
    y, w_event, treat_value, label = get_target_arrays(target_name)
    oof = np.zeros(len(X), dtype=float)
    test_score = np.zeros(len(X_test), dtype=float) if X_test is not None else None
    print(f"\n[{label}] Transformed Outcome {kind}")
    for fold, tr, va in fold_iter(target_name):
        e = np.clip(w_event[tr].mean(), 1e-6, 1 - 1e-6)
        y_star = w_event[tr] * y[tr] / e - (1 - w_event[tr]) * y[tr] / (1 - e)
        m = safe_fit_regressor(kind, X[tr], y_star, seed=SEED + fold)
        pred = predict_regressor(m, X[va])
        oof[va] = pred
        if X_test is not None:
            test_score += predict_regressor(m, X_test) / N_SPLITS
        print_fold_metric(fold, target_name, y, w_event, va, pred)
        del m
        gc.collect()
    full_met = qini_metrics(y, w_event, oof)
    print(f"  OOF: Qini_coefficient_{target_name}={full_met['qini_norm']:+.5f} | Q_ratio={full_met['qini_ratio']:+.4f}")
    return oof, test_score


In [ ]:
def run_x_learner(kind, target_name, X_test=None):
    # X-Learner: T-learner outcome models -> impute individual effects -> effect regressors.
    y, w_event, treat_value, label = get_target_arrays(target_name)
    oof = np.zeros(len(X), dtype=float)
    test_score = np.zeros(len(X_test), dtype=float) if X_test is not None else None
    print(f"\n[{label}] X-Learner {kind}")
    for fold, tr, va in fold_iter(target_name):
        mask_t = w_event[tr] == 1
        mask_c = ~mask_t
        y_t = y[tr][mask_t]
        y_c = y[tr][mask_c]
        spw_common = common_pos_weight(y_t, y_c)
        m_t = safe_fit_classifier(kind, X[tr][mask_t], y_t, class_weights(y_t, spw_common), seed=SEED + fold)
        m_c = safe_fit_classifier(kind, X[tr][mask_c], y_c, class_weights(y_c, spw_common), seed=SEED + 100 + fold)

        d_t = y_t - predict_classifier(m_c, X[tr][mask_t])
        d_c = predict_classifier(m_t, X[tr][mask_c]) - y_c

        tau_t = safe_fit_regressor(kind, X[tr][mask_t], d_t, seed=SEED + 200 + fold)
        tau_c = safe_fit_regressor(kind, X[tr][mask_c], d_c, seed=SEED + 300 + fold)

        e = np.clip(w_event[tr].mean(), 1e-6, 1 - 1e-6)
        pred = e * predict_regressor(tau_c, X[va]) + (1.0 - e) * predict_regressor(tau_t, X[va])
        oof[va] = pred
        if X_test is not None:
            test_score += (e * predict_regressor(tau_c, X_test) + (1.0 - e) * predict_regressor(tau_t, X_test)) / N_SPLITS
        print_fold_metric(fold, target_name, y, w_event, va, pred)
        del m_t, m_c, tau_t, tau_c
        gc.collect()
    full_met = qini_metrics(y, w_event, oof)
    print(f"  OOF: Qini_coefficient_{target_name}={full_met['qini_norm']:+.5f} | Q_ratio={full_met['qini_ratio']:+.4f}")
    return oof, test_score


def run_r_learner(kind, target_name, X_test=None):
    # R-Learner: residualize outcome and randomized treatment; no propensity model is trained.
    y, w_event, treat_value, label = get_target_arrays(target_name)
    oof = np.zeros(len(X), dtype=float)
    test_score = np.zeros(len(X_test), dtype=float) if X_test is not None else None
    print(f"\n[{label}] R-Learner {kind}")
    for fold, tr, va in fold_iter(target_name):
        e = np.clip(w_event[tr].mean(), 1e-6, 1 - 1e-6)
        m_y = safe_fit_classifier(kind, X[tr], y[tr], class_weights(y[tr]), seed=SEED + fold)
        mu_tr = predict_classifier(m_y, X[tr])
        denom = w_event[tr] - e
        denom_safe = np.where(np.abs(denom) < 1e-6, np.where(denom >= 0, 1e-6, -1e-6), denom)
        pseudo_tau = (y[tr] - mu_tr) / denom_safe
        pseudo_w = np.square(denom).astype(np.float32)
        tau_model = safe_fit_regressor(kind, X[tr], pseudo_tau, pseudo_w, seed=SEED + 100 + fold)
        pred = predict_regressor(tau_model, X[va])
        oof[va] = pred
        if X_test is not None:
            test_score += predict_regressor(tau_model, X_test) / N_SPLITS
        print_fold_metric(fold, target_name, y, w_event, va, pred)
        del m_y, tau_model
        gc.collect()
    full_met = qini_metrics(y, w_event, oof)
    print(f"  OOF: Qini_coefficient_{target_name}={full_met['qini_norm']:+.5f} | Q_ratio={full_met['qini_ratio']:+.4f}")
    return oof, test_score


def run_dr_learner(kind, target_name, X_test=None):
    # DR-Learner: doubly-robust pseudo-outcome with constant randomized propensity.
    y, w_event, treat_value, label = get_target_arrays(target_name)
    oof = np.zeros(len(X), dtype=float)
    test_score = np.zeros(len(X_test), dtype=float) if X_test is not None else None
    print(f"\n[{label}] DR-Learner {kind}")
    for fold, tr, va in fold_iter(target_name):
        mask_t = w_event[tr] == 1
        mask_c = ~mask_t
        y_t = y[tr][mask_t]
        y_c = y[tr][mask_c]
        spw_common = common_pos_weight(y_t, y_c)
        m_t = safe_fit_classifier(kind, X[tr][mask_t], y_t, class_weights(y_t, spw_common), seed=SEED + fold)
        m_c = safe_fit_classifier(kind, X[tr][mask_c], y_c, class_weights(y_c, spw_common), seed=SEED + 100 + fold)

        mu1_tr = predict_classifier(m_t, X[tr])
        mu0_tr = predict_classifier(m_c, X[tr])
        e = np.clip(w_event[tr].mean(), 1e-6, 1 - 1e-6)
        phi = (
            mu1_tr - mu0_tr
            + w_event[tr] * (y[tr] - mu1_tr) / e
            - (1 - w_event[tr]) * (y[tr] - mu0_tr) / (1 - e)
        )
        tau_model = safe_fit_regressor(kind, X[tr], phi, seed=SEED + 200 + fold)
        pred = predict_regressor(tau_model, X[va])
        oof[va] = pred
        if X_test is not None:
            test_score += predict_regressor(tau_model, X_test) / N_SPLITS
        print_fold_metric(fold, target_name, y, w_event, va, pred)
        del m_t, m_c, tau_model
        gc.collect()
    full_met = qini_metrics(y, w_event, oof)
    print(f"  OOF: Qini_coefficient_{target_name}={full_met['qini_norm']:+.5f} | Q_ratio={full_met['qini_ratio']:+.4f}")
    return oof, test_score


def extract_uplift_prediction(pred_raw):
    """Chuẩn hóa output từ causalml/utboost về vector uplift 1D."""
    if isinstance(pred_raw, pd.DataFrame):
        num = pred_raw.select_dtypes(include=[np.number])
        for c in ["uplift", "treatment_effect", "maxTreatmentEffect", "recommended_treatment_effect"]:
            if c in num.columns:
                return num[c].to_numpy(dtype=float)
        return num.iloc[:, -1].to_numpy(dtype=float)
    arr = np.asarray(pred_raw)
    if arr.ndim == 1:
        return arr.astype(float)
    if arr.shape[1] >= 2:
        return (arr[:, 1] - arr[:, 0]).astype(float)
    return arr[:, 0].astype(float)


def run_optional_causalml_model(model_type, criterion, target_name, X_test=None):
    # Direct Uplift Tree / Uplift Random Forest / Interaction Tree / Causal Inference Tree.
    if not AVAILABLE.get("causalml", False):
        raise ImportError("causalml is not installed")
    y, w_event, treat_value, label = get_target_arrays(target_name)
    oof = np.zeros(len(X), dtype=float)
    test_score = np.zeros(len(X_test), dtype=float) if X_test is not None else None
    pretty = "UpliftTree" if model_type == "tree" else "UpliftRandomForest"
    print(f"\n[{label}] CausalML {pretty} criterion={criterion}")
    for fold, tr, va in fold_iter(target_name):
        treatment_labels = np.where(w_event == 1, "treatment", "control")
        common_kwargs = dict(
            max_depth=5 if model_type == "forest" else 4,
            min_samples_leaf=120,
            min_samples_treatment=40,
            evaluationFunction=criterion,
            control_name="control",
            random_state=SEED + fold,
        )
        if model_type == "forest":
            model = UpliftRandomForestClassifier(
                n_estimators=80,
                max_features=min(12, X.shape[1]),
                n_jobs=N_JOBS,
                **common_kwargs,
            )
        else:
            model = UpliftTreeClassifier(
                max_features=min(12, X.shape[1]),
                **common_kwargs,
            )
        model.fit(X[tr], treatment=treatment_labels[tr], y=y[tr])
        pred = extract_uplift_prediction(model.predict(X[va]))
        oof[va] = pred
        if X_test is not None:
            test_score += extract_uplift_prediction(model.predict(X_test)) / N_SPLITS
        print_fold_metric(fold, target_name, y, w_event, va, pred)
        del model
        gc.collect()
    full_met = qini_metrics(y, w_event, oof)
    print(f"  OOF: Qini_coefficient_{target_name}={full_met['qini_norm']:+.5f} | Q_ratio={full_met['qini_ratio']:+.4f}")
    return oof, test_score


def run_optional_utboost(target_name, X_test=None):
    if not AVAILABLE.get("utboost", False):
        raise ImportError("utboost is not installed")
    y, w_event, treat_value, label = get_target_arrays(target_name)
    oof = np.zeros(len(X), dtype=float)
    test_score = np.zeros(len(X_test), dtype=float) if X_test is not None else None
    print(f"\n[{label}] UTBoost")
    for fold, tr, va in fold_iter(target_name):
        model = UTBClassifier(
            ensemble_type="boosting",
            criterion="gbm",
            iterations=UTB_ITERATIONS,
            max_depth=4,
        )
        model.fit(X=X[tr], ti=w_event[tr].astype(int), y=y[tr].astype(int))
        pred = extract_uplift_prediction(model.predict(X[va]))
        oof[va] = pred
        if X_test is not None:
            test_score += extract_uplift_prediction(model.predict(X_test)) / N_SPLITS
        print_fold_metric(fold, target_name, y, w_event, va, pred)
        del model
        gc.collect()
    full_met = qini_metrics(y, w_event, oof)
    print(f"  OOF: Qini_coefficient_{target_name}={full_met['qini_norm']:+.5f} | Q_ratio={full_met['qini_ratio']:+.4f}")
    return oof, test_score


## 5. Run all learners + direct uplift models

In [ ]:
scores_oof = {"cash": {}, "card": {}}
scores_test = {"cash": {}, "card": {}}
run_log = []

RUNNER_MAP = {
    "T": run_t_learner,
    "S": run_s_learner,
    "INT": run_interaction_learner,
    "CT": run_class_transformation,
    "TO": run_transformed_outcome,
    "X": run_x_learner,
    "R": run_r_learner,
    "DR": run_dr_learner,
}


def summarize_one_model(model_name):
    row = {"model": model_name}
    if model_name in scores_oof["cash"]:
        row["Qini_coefficient_cash"] = qini_metrics(Y_cash, w_cash, scores_oof["cash"][model_name])["qini_norm"]
    else:
        row["Qini_coefficient_cash"] = np.nan
    if model_name in scores_oof["card"]:
        row["Qini_coefficient_card"] = qini_metrics(Y_card, w_card, scores_oof["card"][model_name])["qini_norm"]
    else:
        row["Qini_coefficient_card"] = np.nan
    row["Qini_coefficient_total"] = row["Qini_coefficient_cash"] + row["Qini_coefficient_card"]
    return pd.DataFrame([row])


def add_pair_result(model_name, cash_tuple, card_tuple):
    cash_oof, cash_te = cash_tuple
    card_oof, card_te = card_tuple
    scores_oof["cash"][model_name] = cash_oof
    scores_oof["card"][model_name] = card_oof
    if cash_te is not None:
        scores_test["cash"][model_name] = cash_te
    if card_te is not None:
        scores_test["card"][model_name] = card_te
    print(f"\n>>> Summary after {model_name}: CASH / CARD / TOTAL")
    display(summarize_one_model(model_name).round(6))


def run_pair(learner_key, kind):
    model_name = f"{learner_key}_{kind.upper()}"
    runner = RUNNER_MAP[learner_key]
    t0 = time.time()
    cash_tuple = runner(kind, "cash", X_test=X_test)
    card_tuple = runner(kind, "card", X_test=X_test)
    add_pair_result(model_name, cash_tuple, card_tuple)
    run_log.append({"model": model_name, "minutes": (time.time() - t0) / 60, "type": "meta_learner"})


def causalml_tree_model_name(model_type, criterion):
    if model_type == "tree" and criterion == "IT":
        return "INTERACTION_TREE"
    if model_type == "tree" and criterion == "CIT":
        return "CAUSAL_INFERENCE_TREE"
    prefix = "UPLIFT_TREE" if model_type == "tree" else "UPLIFT_RF"
    return f"{prefix}_{criterion.upper()}"


def run_direct_causalml_pair(model_type, criterion):
    model_name = causalml_tree_model_name(model_type, criterion)
    t0 = time.time()
    cash_tuple = run_optional_causalml_model(model_type, criterion, "cash", X_test=X_test)
    card_tuple = run_optional_causalml_model(model_type, criterion, "card", X_test=X_test)
    add_pair_result(model_name, cash_tuple, card_tuple)
    run_log.append({"model": model_name, "minutes": (time.time() - t0) / 60, "type": "direct_uplift_causalml"})


def run_utboost_pair():
    model_name = "UTBOOST"
    t0 = time.time()
    cash_tuple = run_optional_utboost("cash", X_test=X_test)
    card_tuple = run_optional_utboost("card", X_test=X_test)
    add_pair_result(model_name, cash_tuple, card_tuple)
    run_log.append({"model": model_name, "minutes": (time.time() - t0) / 60, "type": "direct_utboost"})


start_all = time.time()

print("\n========== RUN META-LEARNER MODELS ==========")
print("Expected meta model names:")
print([f"{lk}_{k.upper()}" for lk in RUN_LEARNERS for k in RUN_MODEL_KINDS if lk in RUNNER_MAP])

for learner_key in RUN_LEARNERS:
    if learner_key not in RUNNER_MAP:
        continue
    for kind in RUN_MODEL_KINDS:
        try:
            run_pair(learner_key, kind)
        except Exception as e:
            model_name = f"{learner_key}_{kind.upper()}"
            print(f"\n!!! Skip {model_name} due to error: {repr(e)}")
            run_log.append({"model": model_name, "minutes": np.nan, "type": "meta_learner", "error": repr(e)})
            gc.collect()

print("\n========== DIRECT UPLIFT TREE/RF DISABLED IN TABNET-ONLY VERSION ==========")
if RUN_OPTIONAL_CAUSALML_TREE:
    if not AVAILABLE.get("causalml", False):
        print("RUN_OPTIONAL_CAUSALML_TREE=True nhưng causalml chưa được cài -> skip direct uplift tree/RF/IT/CIT.")
    else:
        for criterion in CAUSALML_TREE_CRITERIA:
            try:
                run_direct_causalml_pair("tree", criterion)
            except Exception as e:
                model_name = causalml_tree_model_name("tree", criterion)
                print(f"Skip {model_name} due to error: {repr(e)}")
                run_log.append({"model": model_name, "minutes": np.nan, "type": "direct_uplift_causalml", "error": repr(e)})
                gc.collect()
        for criterion in CAUSALML_FOREST_CRITERIA:
            try:
                run_direct_causalml_pair("forest", criterion)
            except Exception as e:
                model_name = causalml_tree_model_name("forest", criterion)
                print(f"Skip {model_name} due to error: {repr(e)}")
                run_log.append({"model": model_name, "minutes": np.nan, "type": "direct_uplift_causalml", "error": repr(e)})
                gc.collect()
else:
    print("RUN_OPTIONAL_CAUSALML_TREE=False -> skip direct uplift tree/RF/IT/CIT.")

print("\n========== UTBOOST DISABLED IN TABNET-ONLY VERSION ==========")
if RUN_OPTIONAL_UTBOOST:
    if not AVAILABLE.get("utboost", False):
        print("RUN_OPTIONAL_UTBOOST=True nhưng utboost chưa được cài -> skip UTBoost.")
    else:
        try:
            run_utboost_pair()
        except Exception as e:
            print(f"Skip UTBOOST due to error: {repr(e)}")
            run_log.append({"model": "UTBOOST", "minutes": np.nan, "type": "direct_utboost", "error": repr(e)})
            gc.collect()
else:
    print("RUN_OPTIONAL_UTBOOST=False -> skip UTBoost.")

print(f"\nTotal model runtime: {(time.time() - start_all)/60:.1f} minutes")
print("Models finished:", list(scores_oof["cash"].keys()))
print("Total finished models:", len(scores_oof["cash"]))


## 6. Summary, fold-mean 5 folds, ensemble

In [ ]:

def summarize_scores_all():
    models = sorted(set(scores_oof["cash"]).union(scores_oof["card"]))
    rows = []
    for model_name in models:
        cash_met = qini_metrics(Y_cash, w_cash, scores_oof["cash"][model_name]) if model_name in scores_oof["cash"] else None
        card_met = qini_metrics(Y_card, w_card, scores_oof["card"][model_name]) if model_name in scores_oof["card"] else None
        row = {
            "model": model_name,
            "Qini_coefficient_cash": cash_met["qini_norm"] if cash_met else np.nan,
            "Qini_coefficient_card": card_met["qini_norm"] if card_met else np.nan,
        }
        row["Qini_coefficient_total"] = row["Qini_coefficient_cash"] + row["Qini_coefficient_card"]
        rows.append(row)
    return pd.DataFrame(rows).sort_values("Qini_coefficient_total", ascending=False).reset_index(drop=True)


def make_fold_qini_table():
    rows = []
    models = sorted(set(scores_oof["cash"]).union(scores_oof["card"]))
    for model_name in models:
        for fold in range(1, N_SPLITS + 1):
            row = {"model": model_name, "fold": fold}
            idx = fold_ids["cash"] == fold
            if model_name in scores_oof["cash"]:
                row["Qini_coefficient_cash"] = qini_metrics(Y_cash[idx], w_cash[idx], scores_oof["cash"][model_name][idx])["qini_norm"]
            else:
                row["Qini_coefficient_cash"] = np.nan
            idx = fold_ids["card"] == fold
            if model_name in scores_oof["card"]:
                row["Qini_coefficient_card"] = qini_metrics(Y_card[idx], w_card[idx], scores_oof["card"][model_name][idx])["qini_norm"]
            else:
                row["Qini_coefficient_card"] = np.nan
            row["Qini_coefficient_total"] = row["Qini_coefficient_cash"] + row["Qini_coefficient_card"]
            rows.append(row)
    return pd.DataFrame(rows).sort_values(["model", "fold"]).reset_index(drop=True)


def make_fold_mean_table(fold_qini_df):
    return (
        fold_qini_df
        .groupby("model", as_index=False)
        .agg(
            n_folds=("fold", "nunique"),
            Mean_Qini_coefficient_cash=("Qini_coefficient_cash", "mean"),
            Mean_Qini_coefficient_card=("Qini_coefficient_card", "mean"),
            Mean_Qini_coefficient_total=("Qini_coefficient_total", "mean"),
        )
        .sort_values("Mean_Qini_coefficient_total", ascending=False)
        .reset_index(drop=True)
    )


# Summary trước ensemble.
summary_before_ensemble = summarize_scores_all()
print("Base model Qini coefficient summary — CASH / CARD / TOTAL")
display(summary_before_ensemble.round(6))

# Rank ensembles: Qini chỉ quan tâm ranking, nên ensemble theo rank thường ổn định hơn average xác suất thô.
base_models = summary_before_ensemble["model"].tolist()
if base_models:
    scores_oof["cash"]["ENS_ALL_RANK"] = rank_average([scores_oof["cash"][m] for m in base_models if m in scores_oof["cash"]])
    scores_oof["card"]["ENS_ALL_RANK"] = rank_average([scores_oof["card"][m] for m in base_models if m in scores_oof["card"]])
    if X_test is not None:
        scores_test["cash"]["ENS_ALL_RANK"] = rank_average([scores_test["cash"][m] for m in base_models if m in scores_test["cash"]])
        scores_test["card"]["ENS_ALL_RANK"] = rank_average([scores_test["card"][m] for m in base_models if m in scores_test["card"]])

    top_models = summary_before_ensemble.head(min(TOPK_ENSEMBLE, len(summary_before_ensemble)))["model"].tolist()
    scores_oof["cash"][f"ENS_TOP{len(top_models)}_RANK"] = rank_average([scores_oof["cash"][m] for m in top_models if m in scores_oof["cash"]])
    scores_oof["card"][f"ENS_TOP{len(top_models)}_RANK"] = rank_average([scores_oof["card"][m] for m in top_models if m in scores_oof["card"]])
    if X_test is not None:
        scores_test["cash"][f"ENS_TOP{len(top_models)}_RANK"] = rank_average([scores_test["cash"][m] for m in top_models if m in scores_test["cash"]])
        scores_test["card"][f"ENS_TOP{len(top_models)}_RANK"] = rank_average([scores_test["card"][m] for m in top_models if m in scores_test["card"]])

    # Weighted ensemble theo Qini total dương trên OOF.
    weights = summary_before_ensemble.set_index("model").loc[top_models, "Qini_coefficient_total"].clip(lower=0).to_numpy()
    if weights.sum() > 0:
        ens_name = f"ENS_TOP{len(top_models)}_WEIGHTED_RANK"
        scores_oof["cash"][ens_name] = rank_average([scores_oof["cash"][m] for m in top_models if m in scores_oof["cash"]], weights=weights)
        scores_oof["card"][ens_name] = rank_average([scores_oof["card"][m] for m in top_models if m in scores_oof["card"]], weights=weights)
        if X_test is not None:
            scores_test["cash"][ens_name] = rank_average([scores_test["cash"][m] for m in top_models if m in scores_test["cash"]], weights=weights)
            scores_test["card"][ens_name] = rank_average([scores_test["card"][m] for m in top_models if m in scores_test["card"]], weights=weights)

summary = summarize_scores_all()
print("Final Qini coefficient summary — CASH / CARD / TOTAL")
display(summary.round(6))

fold_qini_logs = make_fold_qini_table()
print("Fold-level Qini coefficient — CASH / CARD / TOTAL")
display(fold_qini_logs.round(6))

fold_mean_qini_by_model = make_fold_mean_table(fold_qini_logs)
print("Mean Qini coefficient over 5 folds — CASH / CARD / TOTAL")
display(fold_mean_qini_by_model.round(6))

best_model = summary.iloc[0]["model"]
print("Best model by OOF total Qini:", best_model)


## 7. Correlation and Qini curves

In [ ]:

def correlation_table(scores_dict):
    if not scores_dict:
        return pd.DataFrame()
    return pd.DataFrame(scores_dict).corr(method="spearman")

print("Cash score Spearman correlation:")
display(correlation_table(scores_oof["cash"]).round(3))
print("Card score Spearman correlation:")
display(correlation_table(scores_oof["card"]).round(3))


def plot_qini_compare(target_name, y, w_event, score_dict, top_k=6):
    local_rows = []
    for model_name, score in score_dict.items():
        m = qini_metrics(y, w_event, score)
        local_rows.append((model_name, m["qini_norm"]))
    keep_models = [m for m, _ in sorted(local_rows, key=lambda x: x[1], reverse=True)[:top_k]]

    plt.figure(figsize=(10, 6))
    xp, qp = qini_curve(y, w_event, oracle_score(y, w_event))
    ate = qp[-1]
    plt.plot([0, 1], [0, ate], "--", linewidth=1.2, label=f"Random baseline | ATE={ate:+.5f}")
    plt.plot(xp, qp, ":", linewidth=1.4, label="Perfect ranking")

    for model_name in keep_models:
        x, q = qini_curve(y, w_event, score_dict[model_name])
        m = qini_metrics(y, w_event, score_dict[model_name])
        plt.plot(x, q, linewidth=2, label=f"{model_name} | Qini_coeff={m['qini_norm']:+.4f}")

    plt.title(f"Qini Curve — {target_name.upper()}")
    plt.xlabel("Fraction of population sorted by predicted uplift")
    plt.ylabel("Qini")
    plt.grid(alpha=0.3)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

plot_qini_compare("cash", Y_cash, w_cash, scores_oof["cash"])
plot_qini_compare("card", Y_card, w_card, scores_oof["card"])


## 8. Export outputs

In [ ]:
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary_path = OUTPUT_DIR / "qini_summary.csv"
summary.to_csv(summary_path, index=False)
print("Saved:", summary_path)

fold_path = OUTPUT_DIR / "fold_qini_logs.csv"
fold_qini_logs.to_csv(fold_path, index=False)
print("Saved:", fold_path)

fold_mean_path = OUTPUT_DIR / "fold_mean_qini_by_model.csv"
fold_mean_qini_by_model.to_csv(fold_mean_path, index=False)
print("Saved:", fold_mean_path)

run_log_df = pd.DataFrame(run_log)
run_log_path = OUTPUT_DIR / "run_log.csv"
run_log_df.to_csv(run_log_path, index=False)
print("Saved:", run_log_path)

# Save OOF scores từng model.
oof_out = pd.DataFrame()
oof_out[ID_COL if ID_COL else "row_id"] = feat[ID_COL].values if ID_COL else np.arange(len(feat))
oof_out["treatment_cash"] = W
oof_out["cash_sign_30d"] = Y_cash
oof_out["card_sign_30d"] = Y_card
for name, score in scores_oof["cash"].items():
    oof_out[f"cash_{name}"] = score
for name, score in scores_oof["card"].items():
    oof_out[f"card_{name}"] = score

oof_path = OUTPUT_DIR / "oof_model_scores.csv"
oof_out.to_csv(oof_path, index=False)
print("Saved:", oof_path)

# Final train scores lấy theo best model by total Qini.
final_train = pd.DataFrame()
final_train[ID_COL if ID_COL else "row_id"] = feat[ID_COL].values if ID_COL else np.arange(len(feat))
final_train["best_model"] = best_model
final_train["uplift_cash_score"] = scores_oof["cash"][best_model]
final_train["uplift_card_score"] = scores_oof["card"][best_model]
final_train["cash_vs_card_score"] = final_train["uplift_cash_score"] - final_train["uplift_card_score"]
final_train["recommended_strategy"] = np.where(final_train["cash_vs_card_score"] >= 0, "Cash", "Card")

final_train_path = OUTPUT_DIR / "final_scores_train.csv"
final_train.to_csv(final_train_path, index=False)
print("Saved:", final_train_path)

if X_test is not None and best_model in scores_test["cash"] and best_model in scores_test["card"]:
    sub = pd.DataFrame()
    sub[test_id.name if test_id is not None else "row_id"] = test_id.values if test_id is not None else np.arange(len(X_test))
    sub["best_model"] = best_model
    sub["uplift_cash_score"] = scores_test["cash"][best_model]
    sub["uplift_card_score"] = scores_test["card"][best_model]
    sub["cash_vs_card_score"] = sub["uplift_cash_score"] - sub["uplift_card_score"]
    sub["recommended_strategy"] = np.where(sub["cash_vs_card_score"] >= 0, "Cash", "Card")
    sub_path = OUTPUT_DIR / "submission.csv"
    sub.to_csv(sub_path, index=False)
    print("Saved:", sub_path)
else:
    print("No test feature found or best_model has no test score -> submission.csv not created.")

config = {
    "SEED": SEED,
    "N_SPLITS": N_SPLITS,
    "N_JOBS": N_JOBS,
    "RUN_PRESET": RUN_PRESET,
    "RUN_MODEL_KINDS": RUN_MODEL_KINDS,
    "RUN_LEARNERS": RUN_LEARNERS,
    "RUN_OPTIONAL_CAUSALML_TREE": RUN_OPTIONAL_CAUSALML_TREE,
    "RUN_OPTIONAL_UTBOOST": RUN_OPTIONAL_UTBOOST,
    "TABNET_N_D": TABNET_N_D,
    "TABNET_N_A": TABNET_N_A,
    "TABNET_N_STEPS": TABNET_N_STEPS,
    "TABNET_GAMMA": TABNET_GAMMA,
    "TABNET_N_INDEPENDENT": TABNET_N_INDEPENDENT,
    "TABNET_N_SHARED": TABNET_N_SHARED,
    "TABNET_LAMBDA_SPARSE": TABNET_LAMBDA_SPARSE,
    "TABNET_MASK_TYPE": TABNET_MASK_TYPE,
    "TABNET_LR": TABNET_LR,
    "TABNET_WEIGHT_DECAY": TABNET_WEIGHT_DECAY,
    "TABNET_MAX_EPOCHS": TABNET_MAX_EPOCHS,
    "TABNET_PATIENCE": TABNET_PATIENCE,
    "TABNET_BATCH_SIZE": TABNET_BATCH_SIZE,
    "TABNET_VIRTUAL_BATCH_SIZE": TABNET_VIRTUAL_BATCH_SIZE,
    "TABNET_INTERNAL_VAL_SIZE": TABNET_INTERNAL_VAL_SIZE,
    "TABNET_MAX_TRAIN_ROWS": TABNET_MAX_TRAIN_ROWS,
    "TABNET_DEVICE_NAME": TABNET_DEVICE_NAME,
    "TOPK_ENSEMBLE": TOPK_ENSEMBLE,
    "available_packages": {k: bool(v) for k, v in AVAILABLE.items()},
    "finished_models": list(scores_oof["cash"].keys()),
    "best_model": str(best_model),
}
config_path = OUTPUT_DIR / "run_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
print("Saved:", config_path)


## 9. Cách đọc kết quả

- `qini_summary.csv`: bảng tổng hợp Qini OOF của từng model, gồm `Qini_coefficient_cash`, `Qini_coefficient_card`, `Qini_coefficient_total`.
- `fold_qini_logs.csv`: Qini từng fold để kiểm tra độ ổn định.
- `fold_mean_qini_by_model.csv`: trung bình 5 fold của từng model.
- `oof_model_scores.csv`: OOF score của toàn bộ TabNet meta-learner.
- `final_scores_train.csv`: score train theo best model tổng Qini.
- `submission.csv`: chỉ được tạo nếu notebook tìm thấy file test feature.

Preset mặc định:

```python
RUN_PRESET = "TABNET"
RUN_MODEL_KINDS = ["tabnet"]
RUN_LEARNERS = ["T", "S", "INT", "CT", "TO", "X", "R", "DR"]
```

Các model kỳ vọng trong summary:

```text
T_TABNET
S_TABNET
INT_TABNET
CT_TABNET
TO_TABNET
X_TABNET
R_TABNET
DR_TABNET
ENS_ALL_RANK
ENS_TOP*_RANK
ENS_TOP*_WEIGHTED_RANK
```

Ghi chú runtime:

- TabNet fit rất nhiều lần vì mỗi learner chạy 5-fold cho cả cash và card.
- T/X/R/DR còn có nhiều model nội bộ hơn S/INT/CT/TO.
- Nếu notebook quá lâu trên Kaggle, giảm `TABNET_MAX_EPOCHS`, `TABNET_PATIENCE`, hoặc đặt `TABNET_MAX_TRAIN_ROWS=50000`.
- Direct CausalML tree/RF và UTBoost đã tắt vì bản này là TabNet-only.
